# Day 5 — Compute CAGR (1yr, 3yr, 5yr) + Build Comparison Table

Compute CAGR from `nav_history_clean.csv` using:

`CAGR = (NAV_end / NAV_start)^(1/n) - 1`

Then build a comparison table across all funds.

In [1]:
from __future__ import annotations

from pathlib import Path

import numpy as np
import pandas as pd

import plotly.express as px

In [2]:
# --- Repo-root detection ---

_HERE = Path(__file__).resolve() if '__file__' in globals() else Path.cwd()


def _find_repo_root(start: Path) -> Path:
    cand = start
    for _ in range(12):
        if (cand / 'Data' / 'processed' / 'nav_history_clean.csv').exists() and (cand / 'Data' / 'processed' / 'fund_master_clean.csv').exists():
            return cand
        if cand.name == 'notebooks':
            parent = cand.parent
            if (parent / 'Data' / 'processed' / 'nav_history_clean.csv').exists() and (parent / 'Data' / 'processed' / 'fund_master_clean.csv').exists():
                return parent
        cand = cand.parent
    return start.parent


_REPO_ROOT = _find_repo_root(_HERE)
DATA_DIR = _REPO_ROOT / 'Data' / 'processed'

nav_path = DATA_DIR / 'nav_history_clean.csv'
fund_path = DATA_DIR / 'fund_master_clean.csv'

if not nav_path.exists():
    raise FileNotFoundError(f'Missing file: {nav_path.resolve()}')
if not fund_path.exists():
    raise FileNotFoundError(f'Missing file: {fund_path.resolve()}')

print('DATA_DIR:', DATA_DIR.resolve())

DATA_DIR: C:\Mutual Fund Analytics\Data\processed


In [3]:
nav_df = pd.read_csv(nav_path)
fund_df = pd.read_csv(fund_path)

nav_df['date'] = pd.to_datetime(nav_df['date'], errors='coerce')
nav_df['amfi_code'] = pd.to_numeric(nav_df['amfi_code'], errors='coerce').astype('Int64')
nav_df['nav'] = pd.to_numeric(nav_df['nav'], errors='coerce')

fund_df['amfi_code'] = pd.to_numeric(fund_df['amfi_code'], errors='coerce').astype('Int64')

nav_df = nav_df.dropna(subset=['amfi_code', 'date', 'nav']).copy()
fund_df = fund_df.dropna(subset=['amfi_code', 'scheme_name']).copy()

fund_df = fund_df.sort_values('amfi_code')
amfi_to_name = dict(zip(fund_df['amfi_code'].astype(int), fund_df['scheme_name']))

codes = fund_df['amfi_code'].astype(int).tolist()

print('Loaded schemes:', len(codes))
print('NAV rows:', len(nav_df))
print('NAV date range:', nav_df['date'].min(), '->', nav_df['date'].max())

Loaded schemes: 40
NAV rows: 64320
NAV date range: 2022-01-03 00:00:00 -> 2026-05-29 00:00:00


In [4]:
# Choose evaluation window ends from available data
# End date = max date present in nav_df (so CAGR uses latest available NAV).

END_DATE = nav_df['date'].max()

# Use year counts expressed as days for NAV_end/NAV_start mapping
# n years -> n_days approximate with 365.25 for fractional handling

HORIZONS_YEARS = [1, 3, 5]
DAYS_PER_YEAR = 365.25

# Convert horizon years into day deltas
HORIZON_DELTAS = {y: pd.Timedelta(days=int(round(y * DAYS_PER_YEAR))) for y in HORIZONS_YEARS}

print('END_DATE:', END_DATE)
print('HORIZON_DELTAS:', HORIZON_DELTAS)

END_DATE: 2026-05-29 00:00:00
HORIZON_DELTAS: {1: Timedelta('365 days 00:00:00'), 3: Timedelta('1096 days 00:00:00'), 5: Timedelta('1826 days 00:00:00')}


In [5]:
# For each scheme: pick NAV_start at (END_DATE - n_years) or the closest available earlier date
# Because nav_history_clean.csv forward-fills calendar days, exact matching usually exists,
# but we still choose the last available date <= target for robustness.

nav_df = nav_df.sort_values(['amfi_code', 'date']).copy()

# Pre-group for speed
groups = dict(tuple(nav_df.groupby('amfi_code', sort=False)))

rows = []

for amfi_code in codes:
    df = groups.get(amfi_code)
    if df is None or df.empty:
        continue

    df = df[['date', 'nav']].copy()
    # Ensure sorted
    df = df.sort_values('date')

    end_nav = df.loc[df['date'] == END_DATE, 'nav']
    if end_nav.empty:
        # fallback: closest earlier date
        end_nav = df.loc[df['date'] <= END_DATE, 'nav'].iloc[-1:]
        end_date_used = df.loc[df['date'] <= END_DATE, 'date'].iloc[-1]
    else:
        end_date_used = END_DATE
        end_nav = end_nav.iloc[0]

    out = {'amfi_code': int(amfi_code), 'scheme_name': amfi_to_name.get(int(amfi_code), str(amfi_code))}

    for y in HORIZONS_YEARS:
        target = END_DATE - HORIZON_DELTAS[y]
        # last available NAV on/before target
        start_row = df.loc[df['date'] <= target]
        if start_row.empty:
            out[f'cagr_{y}yr_pct'] = np.nan
            continue

        start_date_used = start_row['date'].iloc[-1]
        start_nav = start_row['nav'].iloc[-1]

        # CAGR = (end/start)^(1/n) - 1
        n_years = float(y)
        cagr = (float(end_nav) / float(start_nav)) ** (1.0 / n_years) - 1.0
        out[f'cagr_{y}yr_pct'] = cagr * 100.0

    out['nav_end_date'] = str(end_date_used.date())
    rows.append(out)

comparison_df = pd.DataFrame(rows)

print('Computed rows:', len(comparison_df))
comparison_df.head()

Computed rows: 40


,amfi_code,scheme_name,cagr_1yr_pct,cagr_3yr_pct,cagr_5yr_pct,nav_end_date
0,100016,HDFC Top 100 Fund - Regular Plan - Growth,-2.224271,1.292649,NaN,2026-05-29
1,100025,HDFC Short Term Debt Fund - Regular - Growth,3.704969,3.916390,NaN,2026-05-29
2,100033,HDFC Mid-Cap Opportunities Fund - Regular - Gr...,53.232396,32.442459,NaN,2026-05-29
3,101206,ABSL Frontline Equity Fund - Regular - Growth,47.924120,28.967695,NaN,2026-05-29
4,101207,ABSL Small Cap Fund - Regular - Growth,-23.986032,-4.152381,NaN,2026-05-29


In [6]:
# Basic filtering / formatting
for y in HORIZONS_YEARS:
    col = f'cagr_{y}yr_pct'
    if col in comparison_df.columns:
        comparison_df[col] = pd.to_numeric(comparison_df[col], errors='coerce')

# Comparison table sort by 5yr CAGR
comparison_table = comparison_df.sort_values('cagr_5yr_pct', ascending=False).reset_index(drop=True)

# Save
out_path = DATA_DIR / 'cagr_comparison_1yr_3yr_5yr.csv'
comparison_table.to_csv(out_path, index=False)

print('Wrote:', out_path.resolve())
comparison_table.head(10)

Wrote: C:\Mutual Fund Analytics\Data\processed\cagr_comparison_1yr_3yr_5yr.csv


,amfi_code,scheme_name,cagr_1yr_pct,cagr_3yr_pct,cagr_5yr_pct,nav_end_date
0,100016,HDFC Top 100 Fund - Regular Plan - Growth,-2.224271,1.292649,NaN,2026-05-29
1,100025,HDFC Short Term Debt Fund - Regular - Growth,3.704969,3.916390,NaN,2026-05-29
2,100033,HDFC Mid-Cap Opportunities Fund - Regular - Gr...,53.232396,32.442459,NaN,2026-05-29
3,101206,ABSL Frontline Equity Fund - Regular - Growth,47.924120,28.967695,NaN,2026-05-29
4,101207,ABSL Small Cap Fund - Regular - Growth,-23.986032,-4.152381,NaN,2026-05-29
5,101208,ABSL Liquid Fund - Regular - Growth,7.236645,6.315784,NaN,2026-05-29
6,102885,UTI Nifty 50 Index Fund - Regular - Growth,20.207704,19.667262,NaN,2026-05-29
7,102886,UTI Mid Cap Fund - Regular - Growth,-16.797481,-0.767406,NaN,2026-05-29
8,102887,UTI Flexi Cap Fund - Regular - Growth,13.583135,25.556188,NaN,2026-05-29
9,118632,Nippon India Large Cap Fund - Regular - Growth,33.981048,22.652360,NaN,2026-05-29


In [7]:
# Optional: visualize distributions
for y in HORIZONS_YEARS:
    col = f'cagr_{y}yr_pct'
    fig = px.histogram(comparison_table, x=col, nbins=60, title=f'Distribution of {y}Y CAGR (%)')
    fig.update_layout(template='plotly_white')
    fig.show()